# Питание и рацион - Исследование (recipes.ipynb)

Цель ноутбука: подготовить всё, что затем оформляется в классы и методы модуля `recipes.py` и используется скриптом `nutritionist.py`.

Разделы:
1. Загрузка и подготовка данных (Epicurious dataset)
2. Регрессия (предсказание числового рейтинга)
3. Классификация (предсказание класса рейтинга)
4. Nutrition Facts (USDA FoodData Central API)
5. Similar Recipes (сбор URL)
6. Сохранение артефактов для основной программы

In [1]:
import json
import re
import numpy as np
import pandas as pd
import pickle
import requests
import time
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier,
    ExtraTreesRegressor, ExtraTreesClassifier,
)
from sklearn.metrics import (
    root_mean_squared_error, accuracy_score, precision_score, recall_score,
    make_scorer, classification_report,
)

RANDOM_STATE = 21
np.random.seed(RANDOM_STATE)

## 1. Загрузка и подготовка данных

Используем реальные файлы датасета Epicurious (HugoDarwood):
- `epi_r.csv` - бинарные признаки (679 столбцов: 6 числовых + 674 тега/ингредиента)
- `full_format_recipes.json` - те же рецепты с полным *текстом* ингредиентов (например `"1 cup dried brown lentils"`), который мы используем, чтобы автоматически отличить настоящие ингредиенты от прочих тегов (жанр, повод, место, диета и т.п.)

**Важно про кодировку:** в `epi_r.csv` есть баг автора датасета - два столбца испорчены двойным неправильным декодированием UTF-8 (`bon appétit` и `créme de cacao`). Читаем файл в `latin-1`, чтобы не потерять данные, и вручную схлопываем дубль-колонки.

In [2]:
df = pd.read_csv('data/epi_r.csv', encoding='latin-1')
print(df.shape)
df.head()

(20052, 680)


,title,rating,calories,protein,fat,sodium,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
0,"Lentil, Apple, and Turkey Wrap",2.500,426.0,30.0,7.0,559.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,Boudin Blanc Terrine with Red Onion Confit,4.375,403.0,18.0,23.0,1439.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Potato and Fennel Soup Hodge,3.750,165.0,6.0,7.0,165.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Mahi-Mahi in Tomato Olive Sauce,5.000,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Spinach Noodle Casserole,3.125,547.0,20.0,32.0,452.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
DUPLICATE_COLUMN_PAIRS = [
    ('bon appÃ©tit', 'bon appï¿½ï¿½tit'),
    ('crÃ©me de cacao', 'crï¿½ï¿½me de cacao'),
]
for good, broken in DUPLICATE_COLUMN_PAIRS:
    if good in df.columns and broken in df.columns:
        df[good] = df[[good, broken]].fillna(0).max(axis=1)
        df = df.drop(columns=[broken])
print(df.shape)

(20052, 678)


In [4]:
with open('data/full_format_recipes.json', encoding='utf-8') as f:
    raw_recipes = json.load(f)
print(len(raw_recipes), 'recipes in JSON')
raw_recipes[0]

20130 recipes in JSON


{'directions': ['1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool.',
  '2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper.',
  '3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and serve. If using tortillas, spread the lentils in the center, top with the turkey and lettuce, and fold up the bottom, left side, and right side before rolling away from you.'],
 'fat': 7.0,
 'date': '2006-09-01T04:00:00.000Z',
 'categories': ['Sandwich',
  'Bean',
  'Fruit',
  'Tomato',
  'turkey',

In [5]:
NON_INGREDIENT_COLUMNS = ['title', 'rating', 'calories', 'protein', 'fat', 'sodium']
tag_columns = [c for c in df.columns if c not in NON_INGREDIENT_COLUMNS]

all_ingredient_text = ' | '.join(
    ' '.join(r.get('ingredients') or []) for r in raw_recipes
).lower()

def clean_tag(c):
    c = re.sub(r'[/_-]', ' ', c.lower())
    return re.sub(r'[^a-z0-9 ]', '', c).strip()

ingredient_columns, non_ingredient_tags = [], []
for c in tag_columns:
    cc = clean_tag(c)
    if cc and cc in all_ingredient_text:
        ingredient_columns.append(c)
    else:
        non_ingredient_tags.append(c)

print('Похожи на ингредиенты:', len(ingredient_columns))
print('Похожи на прочие теги (исключаем):', len(non_ingredient_tags))
print(non_ingredient_tags[:30])

Похожи на ингредиенты: 470
Похожи на прочие теги (исключаем): 202
['#cakeweek', '#wasteless', '22-minute meals', '3-ingredient recipes', '30 days of groceries', 'advance prep required', 'alabama', 'anniversary', 'anthony bourdain', 'arizona', 'aspen', 'atlanta', 'australia', 'back to school', 'backyard bbq', 'bastille day', 'beverly hills', 'birthday', 'bon appÃ©tit', 'brunch', 'buffet', 'bulgaria', 'butterscotch/caramel', 'cambridge', 'camping', 'casserole/gratin', 'chicago', 'christmas', 'christmas eve', 'cinco de mayo']


In [6]:
MANUAL_EXCLUDE = [
    # Способы приготовления и кухонные приборы
    'bake', 'boil', 'blender', 'grill', 'roast', 'sauté', 'sautÃ©', 'fry', 'deep-fry', 'pan-fry',
    'simmer', 'freeze/chill', 'chill', 'broil', 'steam', 'poach', 'marinate', 'rub', 'skewer',
    'slow cooker', 'smoker', 'pressure cooker', 'wok', 'microwave', 'food processor',
    'candy thermometer', 'coffee grinder', 'ice cream machine', 'juicer', 'mandoline',
    'mixer', 'mortar and pestle', 'pasta maker', 'ramekin',

    # Категории блюд, время приема пищи и сервировка
    'appetizer', 'dessert', 'dinner', 'breakfast', 'lunch', 'snack', 'side', 'dip', 'drink',
    'cocktail', 'sandwich', 'salad', 'soup', 'stew', 'braise', 'tart', 'pie', 'pizza', 'taco',
    'burrito', 'lasagna', 'muffin', 'pancake', 'waffle', 'cookie', 'cookies', 'brownie',
    'cupcake', 'cake', 'fritter', 'custard', 'sorbet', 'ice cream', 'punch', 'spritzer',
    'potato salad', 'cranberry sauce', 'salad dressing', 'marinade', 'condiment', 'sauce',

    # Праздники, сезоны и мероприятия
    'christmas', 'thanksgiving', 'easter', 'passover', 'kosher for passover', 'halloween',
    'wedding', 'party', 'picnic', 'fall', 'spring', 'summer', 'winter',

    # Географические регионы и города
    'alaska', 'boston', 'brooklyn', 'california', 'canada', 'colorado', 'cuba', 'denver',
    'england', 'florida', 'hawaii', 'idaho', 'israel', 'jamaica', 'japan', 'kansas', 'kentucky',
    'london', 'los angeles', 'louisiana', 'maine', 'maryland', 'mexico', 'new mexico', 'new york',
    'oregon', 'paris', 'pennsylvania', 'peru', 'santa monica', 'spain', 'texas', 'vermont',
    'virginia', 'wisconsin',
    
    # Диетические и рекламные метки
    'fat free', 'healthy', 'kosher', 'low fat', 'low sodium', 'low cholesterol', 'no sugar added',
    'non-alcoholic', 'organic', 'raw', 'vegan', 'vegetarian', 'gourmet', 'leftovers', 'fruit'
]
ingredient_columns = [c for c in ingredient_columns if c not in MANUAL_EXCLUDE]
print('Итоговых ингредиентных признаков:', len(ingredient_columns))

Итоговых ингредиентных признаков: 341


In [7]:
df_clean = df.dropna(subset=['rating']).copy()

X = df_clean[ingredient_columns].fillna(0).astype(int)
y = df_clean['rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

df = X.copy()

if 'jam or jelly' in df.columns and 'jam' not in df.columns:
    df.rename(columns={'jam or jelly': 'jam'}, inplace=True)

X_train.shape, X_test.shape

((16041, 341), (4011, 341))

In [8]:
def fix_encoding(name):
    try:
        return name.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return name
df = df_clean[ingredient_columns].copy()
if 'jam or jelly' in df.columns:
    df.rename(columns={'jam or jelly': 'jam'}, inplace=True)
df.columns = [fix_encoding(col) for col in df.columns]
list(df.columns)

['alcoholic',
 'almond',
 'amaretto',
 'anchovy',
 'anise',
 'aperitif',
 'apple',
 'apple juice',
 'apricot',
 'artichoke',
 'arugula',
 'asian pear',
 'asparagus',
 'avocado',
 'bacon',
 'banana',
 'barley',
 'basil',
 'bass',
 'bean',
 'beef',
 'beef rib',
 'beef shank',
 'beef tenderloin',
 'beer',
 'beet',
 'bell pepper',
 'berry',
 'biscuit',
 'bitters',
 'blackberry',
 'blue cheese',
 'blueberry',
 'bok choy',
 'bourbon',
 'bran',
 'brandy',
 'bread',
 'breadcrumbs',
 'brie',
 'brine',
 'brisket',
 'broccoli',
 'broccoli rabe',
 'brown rice',
 'brussel sprout',
 'buffalo',
 'bulgur',
 'butter',
 'buttermilk',
 'butternut squash',
 'cabbage',
 'calvados',
 'campari',
 'candy',
 'cantaloupe',
 'capers',
 'caraway',
 'cardamom',
 'carrot',
 'cashew',
 'cauliflower',
 'caviar',
 'celery',
 'chambord',
 'champagne',
 'chard',
 'chartreuse',
 'cheddar',
 'cheese',
 'cherry',
 'chestnut',
 'chicken',
 'chickpea',
 'chile',
 'chile pepper',
 'chili',
 'chive',
 'chocolate',
 'cilantro',

## 2. Регрессия

Предсказываем числовой рейтинг по ингредиентам.

In [9]:
def evaluate_regressor(model, params, X_train, y_train, X_test, y_test, cv=5):
    search = GridSearchCV(model, params, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=2)
    search.fit(X_train, y_train)
    best = search.best_estimator_
    preds = best.predict(X_test)
    rmse = root_mean_squared_error(y_test, preds)
    print(f'{model.__class__.__name__}: best_params={search.best_params_}, test RMSE={rmse:.4f}')
    return best, rmse

In [10]:
# >2 базовых алгоритмов
results_reg = {}

results_reg['linear'] = evaluate_regressor(
    LinearRegression(), {}, X_train, y_train, X_test, y_test
)
results_reg['tree'] = evaluate_regressor(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    {'max_depth': [3, 5, 8, 12, None]},
    X_train, y_train, X_test, y_test
)
results_reg['knn'] = evaluate_regressor(
    KNeighborsRegressor(),
    {'n_neighbors': [5, 15, 30]},
    X_train, y_train, X_test, y_test
)

LinearRegression: best_params={}, test RMSE=1.2686
DecisionTreeRegressor: best_params={'max_depth': 5}, test RMSE=1.2844
KNeighborsRegressor: best_params={'n_neighbors': 30}, test RMSE=1.2854


In [11]:
# >2 ансамблей
results_reg['random_forest'] = evaluate_regressor(
    RandomForestRegressor(random_state=RANDOM_STATE),
    {'n_estimators': [100, 300], 'max_depth': [5, 10, None]},
    X_train, y_train, X_test, y_test
)
results_reg['gradient_boosting'] = evaluate_regressor(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1]},
    X_train, y_train, X_test, y_test
)
results_reg['extra_trees'] = evaluate_regressor(
    ExtraTreesRegressor(random_state=RANDOM_STATE),
    {'n_estimators': [100, 300], 'max_depth': [5, 10, None]},
    X_train, y_train, X_test, y_test
)

RandomForestRegressor: best_params={'max_depth': 10, 'n_estimators': 300}, test RMSE=1.2756
GradientBoostingRegressor: best_params={'learning_rate': 0.1, 'n_estimators': 300}, test RMSE=1.2575
ExtraTreesRegressor: best_params={'max_depth': 5, 'n_estimators': 100}, test RMSE=1.2844


In [12]:
naive_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
naive_rmse = root_mean_squared_error(y_test, naive_pred)
print('Naive regressor RMSE:', naive_rmse)

for name, (model, rmse) in results_reg.items():
    print(f'{name}: RMSE={rmse:.4f} (naive={naive_rmse:.4f})')

Naive regressor RMSE: 1.307237065771332
linear: RMSE=1.2686 (naive=1.3072)
tree: RMSE=1.2844 (naive=1.3072)
knn: RMSE=1.2854 (naive=1.3072)
random_forest: RMSE=1.2756 (naive=1.3072)
gradient_boosting: RMSE=1.2575 (naive=1.3072)
extra_trees: RMSE=1.2844 (naive=1.3072)


## 3. Классификация

**Важно:** классы сильно несбалансированы (`great` ~79%, `so-so` ~11%, `bad` ~10%), поэтому во всех классификаторах ниже используем `class_weight='balanced'` (или `'balanced_subsample'` для лесов) - иначе модель просто всегда предсказывает `great` и не имеет смысла.

### 3.1 Классы = округлённый рейтинг (0..5)

In [13]:
y_rounded = df_clean['rating'].round().clip(0, 5).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_rounded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_rounded
)

In [14]:
def evaluate_classifier(model, params, X_train, y_train, X_test, y_test, scoring='accuracy', cv=5):
    search = GridSearchCV(model, params, cv=cv, scoring=scoring, n_jobs=2)
    search.fit(X_train, y_train)
    best = search.best_estimator_
    preds = best.predict(X_test)
    if scoring == 'accuracy':
        score = accuracy_score(y_test, preds)
    else:
        score = search.score(X_test, y_test)
    print(f'{model.__class__.__name__}: best_params={search.best_params_}, test {scoring}={score:.4f}')
    return best, score

In [15]:
# >2 моделей, с перебором гиперпараметров, оцениваем на тесте только лучшую
results_clf_rounded = {}
results_clf_rounded['logreg'] = evaluate_classifier(
    LogisticRegression(max_iter=1000, class_weight='balanced'), {'C': [0.1, 1, 10]},
    Xc_train, yc_train, Xc_test, yc_test
)
results_clf_rounded['tree'] = evaluate_classifier(
    DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    {'max_depth': [3, 5, 8, None]},
    Xc_train, yc_train, Xc_test, yc_test
)
results_clf_rounded['knn'] = evaluate_classifier(
    KNeighborsClassifier(), {'n_neighbors': [5, 15, 30]},
    Xc_train, yc_train, Xc_test, yc_test
)

LogisticRegression: best_params={'C': 0.1}, test accuracy=0.2580
DecisionTreeClassifier: best_params={'max_depth': None}, test accuracy=0.4665
KNeighborsClassifier: best_params={'n_neighbors': 30}, test accuracy=0.6584


In [16]:
naive_class = yc_train.mode()[0]
naive_preds = np.full_like(yc_test, fill_value=naive_class)
naive_acc = accuracy_score(yc_test, naive_preds)
print(f'Naive classifier (most frequent class) accuracy: {naive_acc:.4f}')

for name, (model, acc) in results_clf_rounded.items():
    print(f'{name}: accuracy={acc:.4f} (naive={naive_acc:.4f})')

Naive classifier (most frequent class) accuracy: 0.6577
logreg: accuracy=0.2580 (naive=0.6577)
tree: accuracy=0.4665 (naive=0.6577)
knn: accuracy=0.6584 (naive=0.6577)


### 3.2 Классы = bad / so-so / great (по accuracy)
бинаризации до 3 классов, именно в этих классах работает основная программа.

In [17]:
def to_bad_soso_great(rounded_rating):
    if rounded_rating <= 1:
        return 'bad'
    elif rounded_rating <= 3:
        return 'so-so'
    else:
        return 'great'

y_group = y_rounded.apply(to_bad_soso_great)
Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X, y_group, test_size=0.2, random_state=RANDOM_STATE, stratify=y_group
)
y_group.value_counts(normalize=True)

rating
great    0.793287
so-so    0.106972
bad      0.099741
Name: proportion, dtype: float64

In [18]:
results_clf_group = {}
results_clf_group['logreg'] = evaluate_classifier(
    LogisticRegression(max_iter=1000, class_weight='balanced'), {'C': [0.1, 1, 10]},
    Xg_train, yg_train, Xg_test, yg_test
)
results_clf_group['tree'] = evaluate_classifier(
    DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    {'max_depth': [3, 5, 8, None]},
    Xg_train, yg_train, Xg_test, yg_test
)
results_clf_group['knn'] = evaluate_classifier(
    KNeighborsClassifier(), {'n_neighbors': [5, 15, 30]},
    Xg_train, yg_train, Xg_test, yg_test
)

naive_group_class = yg_train.mode()[0]
naive_group_acc = accuracy_score(yg_test, np.full_like(yg_test, fill_value=naive_group_class))
print('Naive (most frequent class) accuracy:', naive_group_acc)

LogisticRegression: best_params={'C': 0.1}, test accuracy=0.4520
DecisionTreeClassifier: best_params={'max_depth': 3}, test accuracy=0.7240
KNeighborsClassifier: best_params={'n_neighbors': 30}, test accuracy=0.8008
Naive (most frequent class) accuracy: 0.7933183744702069


### 3.3 Выбор метрики

Что хуже: сказать "плохо", когда на деле хорошо, или сказать "хорошо", когда на деле плохо?

Хуже - сказать "great", когда рецепт на деле "bad": пользователь приготовит невкусное блюдо, думая, что оно отличное, и разочаруется в приложении. Нам важно, чтобы среди всех рецептов, которые модель назвала "great", действительно почти все были хорошими - то есть важна **precision класса `great`** (доля реально хороших рецептов среди тех, что модель уверенно назвала отличными).

In [19]:
def great_precision_scorer(y_true, y_pred):
    return precision_score(y_true, y_pred, labels=['great'], average='macro', zero_division=0)

custom_scorer = make_scorer(great_precision_scorer)

results_clf_group_custom = {}
results_clf_group_custom['logreg'] = evaluate_classifier(
    LogisticRegression(max_iter=1000, class_weight='balanced'), {'C': [0.1, 1, 10]},
    Xg_train, yg_train, Xg_test, yg_test, scoring=custom_scorer
)
results_clf_group_custom['tree'] = evaluate_classifier(
    DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    {'max_depth': [3, 5, 8, None]},
    Xg_train, yg_train, Xg_test, yg_test, scoring=custom_scorer
)
results_clf_group_custom['knn'] = evaluate_classifier(
    KNeighborsClassifier(), {'n_neighbors': [5, 15, 30]},
    Xg_train, yg_train, Xg_test, yg_test, scoring=custom_scorer
)

LogisticRegression: best_params={'C': 10}, test make_scorer(great_precision_scorer, response_method='predict')=0.8693
DecisionTreeClassifier: best_params={'max_depth': None}, test make_scorer(great_precision_scorer, response_method='predict')=0.8504
KNeighborsClassifier: best_params={'n_neighbors': 5}, test make_scorer(great_precision_scorer, response_method='predict')=0.8117


### 3.4 Ансамбли с выбранной метрикой (precision класса `great`)

In [20]:
# >2 ансамблей, с class_weight='balanced_subsample' - без этого модель
# почти всегда предсказывает 'great' и не различает плохие комбинации.
results_clf_ensembles = {}
results_clf_ensembles['random_forest'] = evaluate_classifier(
    RandomForestClassifier(class_weight='balanced_subsample', random_state=RANDOM_STATE),
    {'n_estimators': [100, 300], 'max_depth': [10, None]},
    Xg_train, yg_train, Xg_test, yg_test, scoring=custom_scorer
)
results_clf_ensembles['extra_trees'] = evaluate_classifier(
    ExtraTreesClassifier(class_weight='balanced_subsample', random_state=RANDOM_STATE),
    {'n_estimators': [100, 300], 'max_depth': [10, None]},
    Xg_train, yg_train, Xg_test, yg_test, scoring=custom_scorer
)
results_clf_ensembles['gradient_boosting'] = evaluate_classifier(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    {'n_estimators': [100, 300], 'learning_rate': [0.05, 0.1]},
    Xg_train, yg_train, Xg_test, yg_test, scoring=custom_scorer
)

RandomForestClassifier: best_params={'max_depth': None, 'n_estimators': 300}, test make_scorer(great_precision_scorer, response_method='predict')=0.8403
ExtraTreesClassifier: best_params={'max_depth': None, 'n_estimators': 100}, test make_scorer(great_precision_scorer, response_method='predict')=0.8451
GradientBoostingClassifier: best_params={'learning_rate': 0.1, 'n_estimators': 300}, test make_scorer(great_precision_scorer, response_method='predict')=0.8066


## 4. Решение: какую модель сохранить

Сравни RMSE регрессии и метрики классификации выше. Для интерфейса `nutritionist.py` (bad/so-so/great) нужна именно **классификация**, а не регрессия - сохраняем классификатор напрямую в 3 класса.

Данные оборачиваем в `sklearn.Pipeline`, чтобы препроцессинг и модель были одним объектом (это упрощает переиспользование в `recipes.py`).

In [21]:
# Финальный pipeline: препроцессинг (здесь тривиальный - данные уже
# бинарные, но пайплайн оставляет место для будущего масштабирования/
# отбора признаков) + лучшая модель классификации.
# max_depth/n_estimators намеренно ограничены: без этого RandomForest
# на ~450 признаках разрастается в файл на сотни мегабайт, который
# невозможно нормально хранить в git-репозитории.
final_pipeline = Pipeline([
    ('passthrough', 'passthrough'),
    ('classifier', RandomForestClassifier(
        n_estimators=150, max_depth=20,
        class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=2
    )),
])
final_pipeline.fit(Xg_train, yg_train)
print('test precision(great):', great_precision_scorer(yg_test, final_pipeline.predict(Xg_test)))
print('test accuracy:', accuracy_score(yg_test, final_pipeline.predict(Xg_test)))

test precision(great): 0.8536324786324786
test accuracy: 0.6724008975317876


In [22]:
with open('model.pkl', 'wb') as f:
    pickle.dump(final_pipeline, f)
with open('model_columns.pkl', 'wb') as f:
    pickle.dump(ingredient_columns, f)

## 5. Nutrition Facts

Собираем пищевую ценность по каждому ингредиенту через [USDA FoodData Central API](https://fdc.nal.usda.gov/api-guide.html).

In [23]:
import os
USDA_API_KEY = os.environ.get('USDA_API_KEY', '')  # ключ берётся из переменной окружения
USDA_SEARCH_URL = 'https://api.nal.usda.gov/fdc/v1/foods/search'

DAILY_VALUES = {
    # --- Макронутриенты (из файла DRV) ---
    'Protein': 50,                   # g
    'Fat': 78,                       # g
    'Saturated fat': 20,             # g
    'Cholesterol': 300,              # mg
    'Total Carbohydrate': 275,       # g
    'Dietary Fiber': 28,             # g
    'Sodium': 2300,                  # mg
    'Added sugars': 50,              # g

    # --- Витамины и Минералы (из файла RDI) ---
    'Vitamin A': 900,                # mcg RAE
    'Vitamin C': 90,                 # mg
    'Calcium': 1300,                 # mg
    'Iron': 18,                      # mg
    'Vitamin D': 20,                 # mcg
    'Vitamin E': 15,                 # mg
    'Vitamin K': 120,                # mcg
    'Thiamin': 1.2,                  # mg
    'Riboflavin': 1.3,               # mg
    'Niacin': 16,                    # mg NE
    'Vitamin B-6': 1.7,              # mg
    'Folate': 400,                   # mcg DFE
    'Vitamin B-12': 2.4,             # mcg
    'Biotin': 30,                    # mcg
    'Pantothenic acid': 5,           # mg
    'Phosphorus': 1250,              # mg
    'Iodine': 150,                   # mcg
    'Magnesium': 420,                # mg
    'Zinc': 11,                      # mg
    'Selenium': 55,                  # mcg
    'Copper': 0.9,                   # mg
    'Manganese': 2.3,                # mg
    'Chromium': 35,                  # mcg
    'Molybdenum': 45,                # mcg
    'Chloride': 2300,                # mg
    'Potassium': 4700,               # mg
    'Choline': 550,                  # mg
}

def fetch_nutrients(ingredient_name):
    params = {
        'api_key': USDA_API_KEY,
        'query': ingredient_name,
        'pageSize': 1,
        'dataType': ['Foundation', 'SR Legacy'],
    }
    response = requests.get(USDA_SEARCH_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
    foods = data.get('foods', [])
    if not foods:
        return {}
    nutrients = {}
    for n in foods[0].get('foodNutrients', []):
        name = n.get('nutrientName')
        value = n.get('value')
        if name in DAILY_VALUES and value is not None:
            nutrients[name] = value
    return nutrients

In [28]:
nutrition_rows = []
for ingredient in ingredient_columns:
    try:
        raw = fetch_nutrients(ingredient)
    except Exception as e:
        print('skip', ingredient, e)
        raw = {}
    row = {'ingredient': ingredient}
    for nutrient, dv in DAILY_VALUES.items():
        amount = raw.get(nutrient)
        row[nutrient] = round(100 * amount / dv, 1) if amount is not None else None
    nutrition_rows.append(row)
    time.sleep(0.3)  # уважаем rate limit USDA API

nutrition_df = pd.DataFrame(nutrition_rows)
nutrition_df.to_csv('nutrition_facts.csv', index=False)
nutrition_df.head()

,ingredient,Protein,Fat,Saturated fat,Cholesterol,Total Carbohydrate,Dietary Fiber,Sodium,Added sugars,Vitamin A,...,Magnesium,Zinc,Selenium,Copper,Manganese,Chromium,Molybdenum,Chloride,Potassium,Choline
0,alcoholic,1.8,None,None,0.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,almond,52.4,None,None,NaN,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,amaretto,NaN,None,None,NaN,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,anchovy,40.8,None,None,20.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,anise,35.2,None,None,0.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


## 6. Similar Recipes

Строим URL на epicurious.com для каждого рецепта и сохраняем таблицу, по которой затем ищем похожие рецепты по ингредиентам.

**Ограничение исходных данных:** ни `epi_r.csv`, ни `full_format_recipes.json` не содержат сохранённого URL рецепта (сам скрейпер автора датасета, `recipe.py`, ходил по списку URL, полученному отдельно, и он не попал в финальный дамп). Поэтому мы реконструируем URL по названию рецепта (slug), как это делает сам сайт: `.../recipes/food/views/<slug>`. В реальности у epicurious.com в конце slug ещё есть числовой ID (например `-239217`), которого мы восстановить не можем - соответственно ссылка может не открыться напрямую. Это нужно проговорить на защите как осознанное ограничение датасета, а не ошибку реализации.

In [29]:
def slugify(title):
    slug = title.lower().strip()
    slug = re.sub(r"[^a-z0-9\s-]", '', slug)
    slug = re.sub(r"\s+", '-', slug)
    slug = re.sub(r"-+", '-', slug).strip('-')
    return slug

def build_url(title):
    slug = slugify(title)
    if not slug:
        return None
    return f"https://www.epicurious.com/recipes/food/views/{slug}"

In [30]:
similarity_rows = []
for _, row in df_clean.iterrows():
    title = row.get('title')
    if not isinstance(title, str) or not title.strip():
        continue  # пропускаем рецепты без названия
    url = build_url(title)
    if url is None:
        continue  # пропускаем рецепты без URL, как требует задание
    present_ingredients = [col for col in ingredient_columns if row.get(col) == 1]
    if not present_ingredients:
        continue
    similarity_rows.append({
        'title': title,
        'rating': row['rating'],
        'url': url,
        'ingredients': ','.join(present_ingredients),
        # для бонусной части: если есть теги breakfast/lunch/dinner в исходном df - переносим их
        'meal_type': (
            'breakfast' if df.loc[row.name].get('breakfast') == 1 else
            'dinner' if df.loc[row.name].get('dinner') == 1 else
            'lunch' if df.loc[row.name].get('lunch') == 1 else None
        ),
    })

similarity_df = pd.DataFrame(similarity_rows)
similarity_df.to_csv('recipes_similarity.csv', index=False)
similarity_df.head()

,title,rating,url,ingredients,meal_type
0,"Lentil, Apple, and Turkey Wrap",2.500,https://www.epicurious.com/recipes/food/views/...,"apple,bean,lentil,lettuce,tomato,vegetable,turkey",None
1,Boudin Blanc Terrine with Red Onion Confit,4.375,https://www.epicurious.com/recipes/food/views/...,"dried fruit,onion,pork,port",None
2,Potato and Fennel Soup Hodge,3.750,https://www.epicurious.com/recipes/food/views/...,"dairy,fennel,potato,vegetable",None
3,Mahi-Mahi in Tomato Olive Sauce,5.000,https://www.epicurious.com/recipes/food/views/...,"fish,olive,tomato",None
4,Spinach Noodle Casserole,3.125,https://www.epicurious.com/recipes/food/views/...,"cheese,dairy,pasta,vegetable",None


## Итог

После выполнения этого ноутбука в директории должны появиться:
- `model.pkl`, `model_columns.pkl` - для `RatingPredictor`
- `nutrition_facts.csv` - для `NutritionFacts`
- `recipes_similarity.csv` - для `SimilarRecipesFinder`

Эти файлы должны лежать рядом с `recipes.py` и `nutritionist.py`.